In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
!pwd

/oscar/data/epavlick/zyang220/function_vectors/notebooks


In [3]:
import os, re, json
import torch, numpy as np

import sys
sys.path.append('..')
torch.set_grad_enabled(False)

from src.utils.extract_utils import get_mean_head_activations, compute_universal_function_vector
from src.utils.intervention_utils import fv_intervention_natural_text, function_vector_intervention
from src.utils.model_utils import load_gpt_model_and_tokenizer
from src.utils.prompt_utils import load_dataset, word_pairs_to_prompt_data, create_prompt
from src.utils.eval_utils import decode_to_vocab, sentence_eval

In [5]:
!nvidia-smi

Tue Feb 25 08:47:42 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA L40S                    On  | 00000000:61:00.0 Off |                    0 |
| N/A   29C    P8              32W / 350W |     34MiB / 46068MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [6]:
!pwd

/oscar/data/epavlick/zyang220/function_vectors/notebooks


# Load model 

In [7]:
model_name = 'EleutherAI/gpt-j-6b'
model, tokenizer, model_config = load_gpt_model_and_tokenizer(model_name)
EDIT_LAYER = 9 # the layer to add the funciton vector to 

Loading:  EleutherAI/gpt-j-6b


Some weights of the model checkpoint at EleutherAI/gpt-j-6b were not used when initializing GPTJForCausalLM: ['transformer.h.0.attn.bias', 'transformer.h.0.attn.masked_bias', 'transformer.h.1.attn.bias', 'transformer.h.1.attn.masked_bias', 'transformer.h.10.attn.bias', 'transformer.h.10.attn.masked_bias', 'transformer.h.11.attn.bias', 'transformer.h.11.attn.masked_bias', 'transformer.h.12.attn.bias', 'transformer.h.12.attn.masked_bias', 'transformer.h.13.attn.bias', 'transformer.h.13.attn.masked_bias', 'transformer.h.14.attn.bias', 'transformer.h.14.attn.masked_bias', 'transformer.h.15.attn.bias', 'transformer.h.15.attn.masked_bias', 'transformer.h.16.attn.bias', 'transformer.h.16.attn.masked_bias', 'transformer.h.17.attn.bias', 'transformer.h.17.attn.masked_bias', 'transformer.h.18.attn.bias', 'transformer.h.18.attn.masked_bias', 'transformer.h.19.attn.bias', 'transformer.h.19.attn.masked_bias', 'transformer.h.2.attn.bias', 'transformer.h.2.attn.masked_bias', 'transformer.h.20.attn.bi

In [ ]:
model

GPTJForCausalLM(
  (transformer): GPTJModel(
    (wte): Embedding(50400, 4096)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-27): 28 x GPTJBlock(
        (ln_1): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
        (attn): GPTJAttention(
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (out_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): GPTJMLP(
          (fc_in): Linear(in_features=4096, out_features=16384, bias=True)
          (fc_out): Linear(in_features=16384, out_features=4096, bias=True)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (ln_f)

: 

# Load dataset and Compute task-conditioned mean activations

In [9]:
dataset = load_dataset('capitalize', seed=0)
mean_activations = get_mean_head_activations(dataset, model, model_config, tokenizer)

# Compute function vector (FV)

In [10]:
FV, top_heads = compute_universal_function_vector(mean_activations, model, model_config, n_top_heads=10)

# Prompt Creation - ICL, Shuffled-Label, Zero-Shot, and Natural Text

In [8]:
# Sample ICL example pairs, and a test word
dataset = load_dataset('capitalize')
word_pairs = dataset['train'][:5]
test_pair = dataset['test'][21]

prompt_data = word_pairs_to_prompt_data(word_pairs, query_target_pair=test_pair, prepend_bos_token=True)
sentence = create_prompt(prompt_data)
print("ICL prompt:\n", repr(sentence), '\n\n')

shuffled_prompt_data = word_pairs_to_prompt_data(word_pairs, query_target_pair=test_pair, prepend_bos_token=True, shuffle_labels=True)
shuffled_sentence = create_prompt(shuffled_prompt_data)
print("Shuffled ICL Prompt:\n", repr(shuffled_sentence), '\n\n')

zeroshot_prompt_data = word_pairs_to_prompt_data({'input':[], 'output':[]}, query_target_pair=test_pair, prepend_bos_token=True, shuffle_labels=True)
zeroshot_sentence = create_prompt(zeroshot_prompt_data)
print("Zero-Shot Prompt:\n", repr(zeroshot_sentence))

ICL prompt:
 '<|endoftext|>Q: dance\nA: Dance\n\nQ: soda\nA: Soda\n\nQ: before\nA: Before\n\nQ: youthful\nA: Youthful\n\nQ: orange\nA: Orange\n\nQ: turkey\nA:' 


Shuffled ICL Prompt:
 '<|endoftext|>Q: dance\nA: Dance\n\nQ: soda\nA: Soda\n\nQ: before\nA: Youthful\n\nQ: youthful\nA: Before\n\nQ: orange\nA: Orange\n\nQ: turkey\nA:' 


Zero-Shot Prompt:
 '<|endoftext|>Q: turkey\nA:'


# Evaluation

In [9]:
# Perform an intervention on the shuffled setting
clean_logits, interv_logits = function_vector_intervention(
    shuffled_sentence, [test_pair['output']], EDIT_LAYER, FV, model, model_config, tokenizer)

print("Input Sentence:", repr(shuffled_sentence), '\n')
print(f"Input Query: {repr(test_pair['input'])}, Target: {repr(test_pair['output'])}\n")
print("Few-Shot-Shuffled Prompt Top K Vocab Probs:\n", decode_to_vocab(clean_logits, tokenizer, k=5), '\n')
print("Shuffled Prompt+FV Top K Vocab Probs:\n", decode_to_vocab(interv_logits, tokenizer, k=5))

Input Sentence: '<|endoftext|>Q: dance\nA: Dance\n\nQ: soda\nA: Orange\n\nQ: before\nA: Soda\n\nQ: youthful\nA: Before\n\nQ: orange\nA: Youthful\n\nQ: turkey\nA:' 

Input Query: 'turkey', Target: 'Turkey'

Few-Shot-Shuffled Prompt Top K Vocab Probs:
 [(' Turkey', 0.27363), (' Orange', 0.06435), (' turkey', 0.01775), (' T', 0.01769), (' Soda', 0.01689)] 

Shuffled Prompt+FV Top K Vocab Probs:
 [(' Turkey', 0.83459), (' Tur', 0.01468), (' T', 0.01271), (' turkey', 0.01228), (' Turk', 0.00185)]


In [10]:
# Intervention on the zero-shot prompt
clean_logits, interv_logits = function_vector_intervention(zeroshot_sentence, 
    [test_pair['output']], EDIT_LAYER, FV, model, model_config, tokenizer)

print("Input Sentence:", repr(zeroshot_sentence), '\n')
print(f"Input Query: {repr(test_pair['input'])}, Target: {repr(test_pair['output'])}\n")
print("Zero-Shot Top K Vocab Probs:\n", decode_to_vocab(clean_logits, tokenizer, k=5), '\n')
print("Zero-Shot+FV Vocab Top K Vocab Probs:\n", decode_to_vocab(interv_logits, tokenizer, k=5))

Input Sentence: '<|endoftext|>Q: turkey\nA:' 

Input Query: 'turkey', Target: 'Turkey'

Zero-Shot Top K Vocab Probs:
 [(' turkey', 0.08303), (' chicken', 0.0258), (' a', 0.02062), (' Turkey', 0.01092), (' I', 0.0093)] 

Zero-Shot+FV Vocab Top K Vocab Probs:
 [(' Turkey', 0.72399), (' turkey', 0.06748), (' Tur', 0.01101), (' T', 0.01031), ('Turkey', 0.0085)]


In [11]:
sentence = f"The word \"{test_pair['input']}\" means"
co, io = fv_intervention_natural_text(sentence, EDIT_LAYER, FV, model, model_config, tokenizer, max_new_tokens=10)

print("Input Sentence: ", repr(sentence))
print("GPT-J:" , repr(tokenizer.decode(co.squeeze())))
print("GPT-J+FV:", repr(tokenizer.decode(io.squeeze())), '\n')

Input Sentence:  'The word "turkey" means'
GPT-J: 'The word "turkey" means "to be a fool" in Turkish.\n'
GPT-J+FV: 'The word "turkey" means "Turkey" in Turkish.\n\nThe word' 



# test fv_intervention attn_out

In [12]:
from baukit import TraceDict, get_module

In [13]:
function_vector = FV
intervention_idx = -1
edit_layer = 1
inputs = tokenizer(sentence, return_tensors='pt').to(model.device)

In [21]:
print(inputs['input_ids'].shape)
print(FV.shape)

torch.Size([1, 53])
torch.Size([1, 4096])


In [22]:
def patch_function_vector_attn_out(edit_layer, fv_vector, device, idx=-1):
    """
    Pacth fv to replace the attntion output of a specific layer 

    Parameters:
    edit_layer: the layer to perform the FV intervention
    fv_vector: the function vector to add as an intervention, '1 d_model'
    device: device of the model (cuda gpu or cpu)
    idx: the token index to add the function vector at

    Returns:
    add_act: a fuction specifying how to replace a layer's attn output with a function vector  
    """
    def patch_act(output, layer_name):
        current_layer = int(layer_name.split(".")[2])
        if current_layer == edit_layer:
            if isinstance(output, tuple):
                output[0][:, idx] = fv_vector.to(device)
                return output
            else:
                output[:, idx] = fv_vector.to(device)
                return output
        else:
            return output

    return patch_act

In [25]:
intervention_fn = patch_function_vector_attn_out(edit_layer, 
            function_vector.reshape(1, model_config['resid_dim']), 
            model.device, idx=intervention_idx
        )
intervention_layers = model_config['attn_hook_names']

with TraceDict(model, layers=intervention_layers, edit_output=intervention_fn):     
    intervention_output = model(**inputs).logits[:,-1,:] # batch_size x n_tokens x vocab_size, only want last token prediction
    


In [26]:
intervention_output

tensor([[ 7.2346,  4.9351,  5.1805,  ..., -6.5670, -6.6403, -6.6170]],
       device='cuda:0')